# 20 — Siapkan & Verifikasi Data (MOT20 + DanceTrack) — alur WinSCP

**Tidak wajib download.** Kamu memindahkan dataset dari PC rumah ke mesin ini via WinSCP,
lalu notebook ini tinggal:

1. memindai semua folder dataset yang sudah ada di `data/s2/`,
2. **otomatis memilih sumber TERLENGKAP** per sekuens (MOT20: train penuh, BUKAN subset `ablation`),
3. menautkan ke layout kerja `data/s2/mot20/train/` dan `data/s2/dancetrack/val/`
   (symlink lama yang menunjuk sumber salah dihapus otomatis),
4. verifikasi GT ber-ID (WAJIB lulus) + synthesize `seqinfo.ini` bila hilang.

**Cara transfer (sekali, via WinSCP):**

- MOT20: folder `mot20_hf/train` (4 sekuens: `MOT20-01/02/03/05`) → taruh di `data/s2/mot20_hf/`
- DanceTrack: folder val (25 sekuens) → taruh di `data/s2/dancetrack_hf/` (boleh masih di dalam `extracted/val`)

Boleh taruh di folder lain mana pun di bawah `data/s2/` — pemindaian otomatis. 
Cek keutuhan sebelum transfer: MOT20 train penuh = 429/2782/2405/3315 frame (bukan 214/1391/1202/1657).
Lisensi: MOT20 riset-only (motchallenge.net), DanceTrack cc-by-4.0 non-komersial; folder `data/` sudah di-gitignore.


In [ ]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)


### 2.1 Pemindaian kandidat dataset

Cari semua folder berisi `img1/` + `gt/gt.txt` di bawah `data/s2/`. Frame terbanyak per
sekuens = sumber terbaik (MOT20: train penuh > ablation; DanceTrack: tidak ada varian
terpotong). Kalau tidak ada sama sekali, berarti belum transfer — pakai WinSCP dulu.

In [ ]:
import glob
cands = []
for gt in glob.glob(str(DATA / "**" / "gt" / "gt.txt"), recursive=True):
    seq_dir = Path(gt).parent.parent
    if (seq_dir / "img1").exists():
        n = len(list((seq_dir / "img1").glob("*.*")))
        cands.append((seq_dir.name, str(seq_dir), n))
for name, p, n in sorted(cands, key=lambda x: (x[0], -x[2])):
    print(f"{name:16s} frames={n:6d}  {p}")
print("\nKandidat ditemukan:", len(cands))


### 2.2 Auto-link ke layout kerja (tanpa input manual)

Untuk tiap nama sekuens dipilih kandidat dengan **frame terbanyak** (MOT20-01..05 dari train penuh, bukan `ablation`). Sumber saat ini hanya dianggap benar bila **frame-nya sudah maksimum**; kalau masih kurang (mis. `data/s2/mot20/train` berisi folder asli truncated 214/1391/1202/1657), folder lama di-**backup** (rename `*.bad-old`) lalu diganti symlink ke sumber penuh. Backup sengaja tidak dihapus — bersihkan manual nanti kalau data valid.

In [ ]:
# py3.8 compatible
import glob
from pathlib import Path

def scan_candidates():
    best = {}
    for gt in glob.glob(str(DATA / "**" / "gt" / "gt.txt"), recursive=True):
        seq_dir = Path(gt).parent.parent
        if not (seq_dir / "img1").exists():
            continue
        n = len(list((seq_dir / "img1").glob("*.*")))
        key = seq_dir.name
        real = str(seq_dir.resolve())
        if key not in best or n > best[key][0]:
            best[key] = (n, real)
    return best

def relink(wanted_names, dst_root):
    best = scan_candidates()
    dst_root.mkdir(parents=True, exist_ok=True)
    missing = []
    for name in wanted_names:
        if name not in best:
            missing.append(name)
            continue
        n, src = best[name]
        dst = dst_root / name
        dst_frames = len(list((dst / "img1").glob("*.*"))) if (dst / "img1").exists() else -1
        if dst_frames == n:
            # folder/symlink yang ada sudah berisi frame penuh -> biarkan
            print(f"ok    {name:16s} sudah benar ({n} frame)")
            continue
        # perlu diganti: backup dulu yg lama (non-destruktif), lalu taut ke sumber penuh
        if dst.is_symlink():
            dst.unlink()
            print(f"re-link {name:16s} (dulu symlink, {dst_frames} frame)")
        elif dst.exists():
            bak = dst_root / (name + ".bad-old")
            k = 1
            while bak.exists():
                bak = dst_root / ("%s.bad-%d" % (name, k)); k += 1
            dst.rename(bak)
            print(f"backup {name:16s} -> {bak.name} (frame {dst_frames})")
        dst.symlink_to(src, target_is_directory=True)
        print(f"link   {name:16s} -> {src}  ({n} frame)")
    return missing

mot20_names = ["MOT20-01", "MOT20-02", "MOT20-03", "MOT20-05"]
miss = relink(mot20_names, DATA / "mot20" / "train")
if miss:
    print("!! BELUM ADA (transfer via WinSCP dulu):", miss)

dance_names = sorted(n for n in scan_candidates() if n.startswith("dancetrack"))
miss2 = relink(dance_names, DATA / "dancetrack" / "val")
print("\nDanceTrack val:", len(dance_names), "sekuens | belum ada:", miss2)


### 2.3 Verify MOT20 — WAJIB lulus

Frame harus 429/2782/2405/3315 (total 8.931 = sama dengan baseline OC-SORT). Kalau masih
214/1391/1202/1657, sumber yang ter-link masih subset `ablation` — cek hasil sel auto-link
di atas.

In [ ]:
!python $S2_ROOT/scripts/data_prep/verify_mot_dataset.py $S2_DATA/mot20/train


### 2.4 Verify DanceTrack — WAJIB lulus

In [ ]:
!python $S2_ROOT/scripts/data_prep/verify_mot_dataset.py $S2_DATA/dancetrack/val --min-sequences 20


### 2.5 seqinfo.ini — synthesize bila hilang

DiffMOT (`info_dir`) dan TrackEval butuh `seqinfo.ini` per sekuens
(imWidth/imHeight/imExt/seqLength).

In [ ]:
import cv2
from pathlib import Path
def synth_seqinfo(seq_dir: Path):
    ini = seq_dir / "seqinfo.ini"
    if ini.exists():
        return False
    imgs = sorted((seq_dir / "img1").glob("*.*"))
    if not imgs:
        print("!! tidak ada img1:", seq_dir); return False
    h, w = cv2.imread(str(imgs[0])).shape[:2]
    ext = imgs[0].suffix
    ini.write_text(
        f"[Sequence]\nname={seq_dir.name}\nimDir=img1\nframeRate=30\n"
        f"seqLength={len(imgs)}\nimWidth={w}\nimHeight={h}\nimExt={ext}\n"
    )
    print("synthesize", seq_dir.name, f"{w}x{h} x{len(imgs)}"); return True

n = 0
for seq_dir in sorted(DATA.glob("mot20/train/*")) + sorted(DATA.glob("dancetrack/val/*")):
    if seq_dir.is_dir():
        n += synth_seqinfo(seq_dir)
print("total synthesize:", n)


### 2.6 Ringkasan — cek semua sekuens siap dipakai

In [ ]:
for split_root in [DATA/"mot20"/"train", DATA/"dancetrack"/"val"]:
    if not split_root.exists():
        print("!! belum ada:", split_root); continue
    for seq in sorted(p for p in split_root.iterdir() if p.is_dir()):
        n = len(list((seq/"img1").glob("*.*")))
        has_gt = (seq/"gt"/"gt.txt").exists()
        has_ini = (seq/"seqinfo.ini").exists()
        print(f"{seq.name:16s} frames={n:6d} gt={has_gt} seqinfo={has_ini}")
print("\nSELESAI — lanjut ke 30_s2_gen_detections.ipynb")


**Lanjut**: `30_s2_gen_detections.ipynb` — generate deteksi dengan bobot fine-tune Skenario A
(`data/s2/weights/best.pt` harus sudah ada).

---

## Opsional — Download dari HF (hanya untuk mesin baru yang TIDAK pakai WinSCP)

Kalau data sudah ditransfer via WinSCP, **lewati seluruh bagian ini** dan langsung ke notebook 30.
Bagian ini hanya cadangan untuk menyiapkan data dari nol di mesin lain.

In [ ]:
import os
# Token Read gratis: https://huggingface.co/settings/tokens (New token -> type Read)
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = "hf_xxx"   # ganti dengan token kamu
# repo MOT20 = ~13 ribu file; disable Xet supaya tidak kena rate limit 429 (download CDN langsung)
os.environ["HF_HUB_DISABLE_XET"] = "1"
print("HF_HUB_DISABLE_XET:", os.environ.get("HF_HUB_DISABLE_XET"))


In [ ]:
# MOT20 — HF Lekim89/MOT20 (train penuh; test/* di-skip)
from huggingface_hub import snapshot_download
snapshot_download(repo_id="Lekim89/MOT20", repo_type="dataset",
                  local_dir=str(DATA/"mot20_hf"), ignore_patterns=["test/*"])


In [ ]:
# DanceTrack — HF noahcao/dancetrack (val saja)
from huggingface_hub import snapshot_download
snapshot_download(repo_id="noahcao/dancetrack", repo_type="dataset",
                  local_dir=str(DATA/"dancetrack_hf"), ignore_patterns=["test/*", "test*", "train*", "*.xlsx"])


In [ ]:
import zipfile
src_val = DATA / "dancetrack_hf" / "val"
if not src_val.is_dir():
    z = DATA / "dancetrack_hf" / "val.zip"
    if z.exists():
        print("extract", z)
        src_val.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(src_val)
        print("selesai extract — jalankan ulang sel '2.2 Auto-link' di atas")
    else:
        print("val.zip tidak ada — cek folder dancetrack_hf/")
